# 00 Overview: what the code does

The code follows the paper's claim that **the test runs on field data only**. Data generation is in a separate
notebook. It writes three kinds of file, and each later notebook reads only what it is allowed to:

![process](process.png)

| Folder | Contents | Who may read it |
|---|---|---|
| `data/measurements/` | what a field team measures: excitations $\mathbf M_d$ and their event days $\bm\Pi_d$, outputs $\mathbf Y_o$ with masks of the measured entries, the canal schedule, shared series $\bm\Omega$ | 02, 03, 04 |
| `data/sketch/` | the channel sketch: **which** entries of $\mathbf\Phi_i$, $\mathbf H_i$, $\mathbf\Gamma_i$, $\mathbf E_i$ are non-zero (no values) | 02, 03, 04 |
| `data/truth/` | the networks $\mathbf W$, $\mathbf P$ and the full matrices | **04 only** (scoring) |
| `results/` | what the test reported | 04 |


## Notebooks and the paper's results

| Notebook | Task | Paper results used |
|---|---|---|
| `01_generate_data` | build random communities, simulate day by day, split into measurements / sketch / truth | model (1)-(2), Appendix C |
| `02_certify_links` | find links from the data | Lemma 1 (lags from the sketch); Step 1 and C2 (residuals, eq. 4); Steps 2-3 (eq. 5, by hand and in matrix form); Theorem 2 (threshold eq. 6, decisions, p-values); Proposition 3 (lag scan without the sketch) |
| `03_what_to_measure` | pilot on every candidate channel; decide what to measure | Corollary 4 (channel certificates); eq. (7) scaled from the pilot; the Procedure of Section 3 |
| `04_score_and_compare` | score against the truth; baselines; predicted vs observed power; wrong sketch | checks Theorem 2, eq. (7), Proposition 3 |
| `05_hand_example` | 3 farms, 2 channels, 50 days, end to end | all of the above, small |

## The code underneath (`lib/`)
* `community.py`: the simulator (farm model, networks, the channel sketch, first lags from the sketch).
* `certify.py`: the test (`residuals`, `test_excitation`, `certify_plan`, `threshold`, `p_values`), the lag scan (`lag_scan`), channel certificates (`channel_certificates`) and the baselines (`baselines`, `conditional_granger`).
* `plans.py`: the measurement plans of Table 3, `view()` (the data a plan provides) and the lags read from a sketch.
* `farmlib.py`: saving and loading the three kinds of file. `scoring.py` and `netplot.py`: scoring and drawing, used only where the truth may be read.

These are the same modules that produced the paper's numbers (`paper_runs/`).

## Run order and run time
Run 01, then 02, 03 and 04 (05 is independent). With the defaults (10 communities, 5 years) the set runs in a
few minutes. The paper's full runs (40 communities, up to 10 years) are in `paper_runs/`.

## One limit the code makes visible
The Procedure predicts each plan from a pilot. From pilot data alone one can estimate each pair's
$\hat\mu(T)=|Z^{\rm pilot}|\sqrt{T/T_{\rm pilot}}$ and scale it to longer durations, which is what 03 does, but not
separate the link weight $S_{ij}$ from the gain $g$ (Appendix B, Status of the predictions). The paper's predicted
power, which uses the **true** gains, is therefore a check of eq. (7) (done in 04), not something a field team
can compute.
